## Data Cleaning

In [1]:
import pandas as pd
import numpy as np
import matplotlib as plt
from sklearn.preprocessing import OneHotEncoder
import datetime
import h3
import geopandas
import json

In [2]:
df = pd.read_csv("../csv/Taxi_Trips_(2024-).csv")

C:\Users\angel\AppData\Local\Temp\ipykernel_28444\3810927188.py:1: DtypeWarning: Columns (0: Trip Miles) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../csv/Taxi_Trips_(2024-).csv")


### 1.1 Converting data

Converting numerical data to numerical data type

In [ ]:
df["Trip Miles"] = df["Trip Miles"].str.replace(".", "", regex=False)
df["Trip Miles"] = df["Trip Miles"].str.replace(",", ".", regex=False)
df["Trip Miles"] = pd.to_numeric(df["Trip Miles"])

In [4]:
df["Extras"] = df["Extras"].str.replace(".", "", regex=False)
df["Extras"] = df["Extras"].str.replace(",", ".", regex=False)
df["Extras"] = df["Extras"].str.replace("$", "", regex=False)
df["Extras"] = pd.to_numeric(df["Extras"])
df["Tips"] = df["Tips"].str.replace(".", "", regex=False)
df["Tips"] = df["Tips"].str.replace(",", ".", regex=False)
df["Tips"] = df["Tips"].str.replace("$", "", regex=False)
df["Tips"] = pd.to_numeric(df["Tips"])
df["Tolls"] = df["Tolls"].str.replace(".", "", regex=False)
df["Tolls"] = df["Tolls"].str.replace(",", ".", regex=False)
df["Tolls"] = df["Tolls"].str.replace("$", "", regex=False)
df["Tolls"] = pd.to_numeric(df["Tolls"])
df["Fare"] = df["Fare"].str.replace(".", "", regex=False)
df["Fare"] = df["Fare"].str.replace(",", ".", regex=False)
df["Fare"] = df["Fare"].str.replace("$", "", regex=False)
df["Fare"] = pd.to_numeric(df["Fare"])

In [5]:
df["Trip Total"] = df["Trip Total"].str.replace(".", "", regex=False)
df["Trip Total"] = df["Trip Total"].str.replace(",", ".", regex=False)
df["Trip Total"] = df["Trip Total"].str.replace("$", "", regex=False)
df["Trip Total"] = pd.to_numeric(df["Trip Total"])

In [19]:
if isinstance(df["Trip Seconds"].dtype, pd.StringDtype) :
    df["Trip Seconds"] = df["Trip Seconds"].str.replace(".", "", regex=False)
    df["Trip Seconds"] = df["Trip Seconds"].str.replace(",", ".", regex=False)
    df["Trip Seconds"] = pd.to_numeric(df["Trip Seconds"])

Change Longitude and Latitude data to numerical data type

In [ ]:
if isinstance(df["Pickup Centroid Latitude"].dtype, pd.StringDtype) :
    df["Pickup Centroid Latitude"] = df["Pickup Centroid Latitude"].str.replace(",", ".")
    df["Pickup Centroid Latitude"] = pd.to_numeric(df["Pickup Centroid Latitude"])

    df["Pickup Centroid Longitude"] = df["Pickup Centroid Longitude"].str.replace(",", ".")
    df["Pickup Centroid Longitude"] = pd.to_numeric(df["Pickup Centroid Longitude"])

    df["Dropoff Centroid Latitude"] = df["Dropoff Centroid Latitude"].str.replace(",", ".")
    df["Dropoff Centroid Latitude"] = pd.to_numeric(df["Dropoff Centroid Latitude"])

    df["Dropoff Centroid Longitude"] = df["Dropoff Centroid Longitude"].str.replace(",", ".")
    df["Dropoff Centroid Longitude"] = pd.to_numeric(df["Dropoff Centroid Longitude"])

Convert date data to datetime

In [7]:
df["Trip Start Timestamp"] = pd.to_datetime(df["Trip Start Timestamp"])
df["Trip End Timestamp"] = pd.to_datetime(df["Trip End Timestamp"])

C:\Users\angel\AppData\Local\Temp\ipykernel_28444\2590110223.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["Trip Start Timestamp"] = pd.to_datetime(df["Trip Start Timestamp"])
C:\Users\angel\AppData\Local\Temp\ipykernel_28444\2590110223.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["Trip End Timestamp"] = pd.to_datetime(df["Trip End Timestamp"])


### 1.2 Cleaning data 

Delete all trips with time seconds < 60

In [20]:
n_before = len(df)
df = df[df["Trip Seconds"] >= 60]
print(f"Dropped {n_before - len(df):,} trips below 60 seconds.")

Dropped 7,874,117 trips below 60 seconds.


Delete all trips with 0 total pay

In [21]:
n_before = len(df)
df = df[df["Trip Total"] != 0]
print(f"Dropped {n_before - len(df):,} trips with no income.")

Dropped 13,726 trips with no income.


Check if df has a location

In [22]:
df = df[df["Pickup Census Tract"].notnull() | df["Pickup Community Area"].notnull() | df["Pickup Centroid Location"].notnull()]

If Payment Type = Null change to unknown

In [23]:
print(df["Payment Type"].unique())
df["Payment Type"] = df["Payment Type"].fillna("Unknown")

<StringArray>
[       'Cash', 'Credit Card',      'Mobile',     'Unknown',      'Prcard',
   'No Charge',     'Dispute',     'Prepaid']
Length: 8, dtype: str


Drop duplicates

In [24]:
n_before = len(df)
# drop duplicates in Trip ID
df = df.drop_duplicates(subset="Trip ID")
print(f"Dropped {n_before - len(df):,} duplicate rows.")

Dropped 0 duplicate rows.


Drop implausible trips

In [25]:
n_before = len(df)
df = df[df["Trip End Timestamp"] > df["Trip Start Timestamp"]]
print(f"Dropped {n_before - len(df):,} nonsensical data. ")

Dropped 2,875,319 nonsensical data. 


In [31]:
df.info()

<class 'pandas.DataFrame'>
Index: 4512772 entries, 5 to 15406956
Data columns (total 32 columns):
 #   Column                      Dtype         
---  ------                      -----         
 0   Trip ID                     str           
 1   Taxi ID                     str           
 2   Trip Start Timestamp        datetime64[us]
 3   Trip End Timestamp          datetime64[us]
 4   Trip Seconds                float64       
 5   Trip Miles                  float64       
 6   Pickup Census Tract         float64       
 7   Dropoff Census Tract        float64       
 8   Pickup Community Area       float64       
 9   Dropoff Community Area      float64       
 10  Fare                        float64       
 11  Tips                        float64       
 12  Tolls                       float64       
 13  Extras                      float64       
 14  Trip Total                  float64       
 15  Company                     str           
 16  Pickup Centroid Latitude    float

## Preparing Data

One hot encode Payment Type

In [26]:
encoder = OneHotEncoder(sparse_output=False)
one_hot_encoded = encoder.fit_transform(df[['Payment Type']])

one_hot_df = pd.DataFrame(
    one_hot_encoded,
    columns=encoder.get_feature_names_out(['Payment Type']),
    index=df.index
)
df = pd.concat([df, one_hot_df], axis=1)
df = df.drop('Payment Type', axis=1)

Converting location to hexagons

In [27]:
# there is no trip where Pickup Census Tract has data, where Pickup Centroid Location does not have
#print(df[df["Pickup Census Tract"].notnull() & df["Pickup Centroid Location"].isnull()].count())
#print(df[df["Pickup Census Tract"].notnull() & df["Pickup Centroid Latitude"].isnull()].count())
#print(df[df["Pickup Census Tract"].notnull() & df["Pickup Centroid Longitude"].isnull()].count())
print(df[df["Pickup Community Area"].notnull() & df["Pickup Centroid Longitude"].isnull()].count())
print(df[df["Dropoff Community Area"].notnull() & df["Dropoff Centroid Longitude"].isnull()].count())

Trip ID                       0
Taxi ID                       0
Trip Start Timestamp          0
Trip End Timestamp            0
Trip Seconds                  0
Trip Miles                    0
Pickup Census Tract           0
Dropoff Census Tract          0
Pickup Community Area         0
Dropoff Community Area        0
Fare                          0
Tips                          0
Tolls                         0
Extras                        0
Trip Total                    0
Company                       0
Pickup Centroid Latitude      0
Pickup Centroid Longitude     0
Pickup Centroid Location      0
Dropoff Centroid Latitude     0
Dropoff Centroid Longitude    0
Dropoff Centroid  Location    0
Payment Type_Cash             0
Payment Type_Credit Card      0
Payment Type_Dispute          0
Payment Type_Mobile           0
Payment Type_No Charge        0
Payment Type_Prcard           0
Payment Type_Prepaid          0
Payment Type_Unknown          0
dtype: int64
Trip ID                    

City boundry block for chicago?

In [28]:
#city_bounding_box = geopandas.read_file('')
#city_bounding_box_json_string = city_bounding_box.to_json()
#city_bounding_box_json = json.loads(city_bounding_box_json_string)
#city_bounding_box_poly = city_bounding_box_json["features"][0]

In [29]:
resolution = 7

In [30]:

df["hex_loc_pick_up"] = df.apply(
    lambda row: h3.latlng_to_cell(
        row["Pickup Centroid Latitude"],
        row["Pickup Centroid Longitude"],
        resolution
    )
    if pd.notna(row["Pickup Centroid Latitude"]) and pd.notna(row["Pickup Centroid Longitude"])
    else 0,
    axis=1
)

df["hex_loc_drop_off"] = df.apply(
    lambda row: h3.latlng_to_cell(
        row["Dropoff Centroid Latitude"],
        row["Dropoff Centroid Longitude"],
        resolution
    )
    if pd.notna(row["Dropoff Centroid Latitude"]) and pd.notna(row["Dropoff Centroid Longitude"])
    else 0,
    axis=1
)